# CSIRO Biomass image inference

Generate `submission.csv` from a locally trained model uploaded to a Kaggle Dataset.

In [ ]:
import os
import sys
import glob
import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel, CLIPProcessor, CLIPModel

# ==========================================
# 設定
# ==========================================
COMP_DIR = "/kaggle/input/csiro-biomass"
DATASET_DIR = "/kaggle/input/koro2jp"

if DATASET_DIR not in sys.path:
    sys.path.append(DATASET_DIR)

TARGET_NAMES = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']
MAX_VALS = np.array([71.7865, 83.8407, 157.9836, 185.70, 157.9836])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ==========================================
# 1. モデルロード関数
# ==========================================
def load_model_from_input(target_name):
    print(f"Searching model for: {target_name} ...")
    
    # CLIP検索ロジック
    if "clip" in target_name.lower():
        patterns = [
            "/kaggle/input/*openai*clip*vit*large*336*",
            "/kaggle/input/*clip*vit*large*336*",
            "/kaggle/input/*clip*vit*large*",
            "/kaggle/input/*openai*clip*"
        ]
        candidates = []
        for pat in patterns:
            candidates.extend(glob.glob(pat))
        
        for path in candidates:
            if os.path.exists(os.path.join(path, "config.json")) or \
               glob.glob(os.path.join(path, "*", "config.json")):
                if not os.path.exists(os.path.join(path, "config.json")):
                    sub = glob.glob(os.path.join(path, "*", "config.json"))
                    if sub: path = os.path.dirname(sub[0])
                print(f"Found CLIP at: {path}")
                return path
        raise FileNotFoundError("CLIP model dataset not found in Input!")

    # その他モデル検索
    candidates = glob.glob(f"/kaggle/input/*{target_name}*")
    model_path = None
    for path in candidates:
        if os.path.exists(os.path.join(path, "config.json")):
            model_path = path; break
        sub = glob.glob(os.path.join(path, "*", "config.json"))
        if sub: model_path = os.path.dirname(sub[0]); break
    
    if model_path is None: 
        raise FileNotFoundError(f"Model {target_name} not found.")
        
    print(f"Loading {target_name} from: {model_path}")
    return model_path

# ==========================================
# 2. 特徴量抽出関数 (TTA実装版: 通常 + 左右反転)
# ==========================================
def get_embeddings(image_paths, model_path, model_type="siglip", batch_size=32):
    print(f"Loading {model_type} model from {model_path}...")
    
    if model_type == "clip":
        processor = CLIPProcessor.from_pretrained(model_path)
        model = CLIPModel.from_pretrained(model_path).to(DEVICE).eval()
    else:
        processor = AutoImageProcessor.from_pretrained(model_path)
        model = AutoModel.from_pretrained(model_path).to(DEVICE).eval()

    embeddings = []
    
    # TTA: 通常画像と反転画像の2回ループするのではなく、バッチ内で処理する
    for i in tqdm(range(0, len(image_paths), batch_size), desc=f"Extract {model_type} (w/ TTA)"):
        batch_paths = image_paths[i : i + batch_size]
        images = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except:
                # エラー時は黒画像
                images.append(Image.new("RGB", (224, 224)))
        
        if not images: continue
        
        # --- TTA処理 ---
        # 1. 元画像
        # 2. 左右反転画像
        images_flipped = [img.transpose(Image.FLIP_LEFT_RIGHT) for img in images]
        
        # 処理用にリストを結合して一気に推論
        combined_images = images + images_flipped
        
        with torch.no_grad():
            if model_type == "clip":
                inputs = processor(images=combined_images, return_tensors="pt", padding=True).to(DEVICE)
                out = model.get_image_features(**inputs)
            else:
                inputs = processor(images=combined_images, return_tensors="pt").to(DEVICE)
                if model_type == "siglip":
                    out = model.get_image_features(**inputs)
                else: 
                    outputs = model(**inputs)
                    out = outputs.last_hidden_state[:, 0, :] if hasattr(outputs, "last_hidden_state") else outputs[0][:, 0, :]

        # 正規化
        emb = out / out.norm(dim=-1, keepdim=True)
        emb = emb.cpu().numpy()
        
        # 前半(通常)と後半(反転)に分割して平均をとる
        n = len(images)
        emb_original = emb[:n]
        emb_flipped = emb[n:]
        emb_avg = (emb_original + emb_flipped) / 2.0
        
        embeddings.append(emb_avg)
        
    return np.vstack(embeddings)

# ==========================================
# 3. 物理整合性後処理
# ==========================================
def post_process(df):
    cols = ["Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g", "GDM_g", "Dry_Total_g"]
    Y = df[cols].values.T
    C = np.array([[1,1,0,-1,0], [0,0,1,1,-1]])
    P = np.eye(5) - C.T @ np.linalg.inv(C @ C.T) @ C
    Y_rec = (P @ Y).T.clip(min=0)
    df[cols] = Y_rec
    return df

# ==========================================
# Main: 推論実行
# ==========================================
# 1. 画像リスト取得
test_df = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
unique_images = test_df[['image_path']].drop_duplicates().reset_index(drop=True)
full_paths = [os.path.join(COMP_DIR, p) for p in unique_images['image_path']]
print(f"Test Images: {len(full_paths)}")

# 2. 特徴量抽出
print("Extracting features...")
sig_path = load_model_from_input("siglip")
emb_sig = get_embeddings(full_paths, sig_path, "siglip") # 1152

dino_path = load_model_from_input("dinov2")
emb_dino = get_embeddings(full_paths, dino_path, "dinov2") # 768

clip_path = load_model_from_input("clip") 
emb_clip = get_embeddings(full_paths, clip_path, "clip") # 768

print("Features extracted.")

# 3. モデル (LGBM, XGB, Cat) の推論
models_to_run = ["lgbm", "xgb", "cat"]
preds_total = np.zeros((len(emb_sig), 5))
valid_models = 0

print("-" * 30)
print("Start Predicting with Smart Feature Matching (No Semantic)...")

for keyword in models_to_run:
    # モデルフォルダ検索
    pat = os.path.join(DATASET_DIR, "**", f"*{keyword}*", "models_fold_*.pkl")
    files = glob.glob(pat, recursive=True)
    valid_dirs = list(set([os.path.dirname(f) for f in files]))
    
    if not valid_dirs:
        print(f"⚠️ Skip {keyword}: No model folder found.")
        continue
        
    for model_dir in valid_dirs:
        folder_name = os.path.basename(model_dir)
        print(f"\n--- Predicting with {folder_name} ---")
        
        pred_accum = np.zeros((len(emb_sig), 5))
        folds = 0
        success_folds = 0
        
        for fold in range(5):
            ep = os.path.join(model_dir, f"engine_fold_{fold}.pkl")
            mp = os.path.join(model_dir, f"models_fold_{fold}.pkl")
            if not os.path.exists(ep) or not os.path.exists(mp): continue
            
            try:
                eng = joblib.load(ep)
                ms = joblib.load(mp)
                
                # --- スマート次元合わせ (Update for Large) ---
                if hasattr(eng, "scaler") and hasattr(eng.scaler, "n_features_in_"):
                    expected_dim = eng.scaler.n_features_in_
                else:
                    expected_dim = 2944
                
                # 次元数に応じて、正しいパーツだけを合体させる
                if expected_dim == 2944:
                    # Large (SigLIP 1152 + DINOv2_L 1024 + CLIP 768)
                    X_in = np.hstack([emb_sig, emb_dino, emb_clip])

                elif expected_dim == 2688:
                    # Base (SigLIP 1152 + DINOv2_B 768 + CLIP 768)
                     X_in = np.hstack([emb_sig, emb_dino, emb_clip])

                elif expected_dim == 1920:
                    # Basic (SigLIP + DINO)
                    X_in = np.hstack([emb_sig, emb_dino])
                
                elif expected_dim == 1152:
                    # SigLIPのみ
                    X_in = emb_sig
                    
                else:
                    # 想定外の場合は、全部つなげてスライス
                    print(f"    ⚠️ Unknown dim {expected_dim}. Trying simple slice.")
                    X_temp = np.hstack([emb_sig, emb_dino, emb_clip])
                    X_in = X_temp[:, :expected_dim]

                # 推論
                X_eng = eng.transform(X_in)
                p_fold = []
                for i, m in enumerate(ms):
                    if getattr(m, "_n_classes", None) is None: m._n_classes = 1
                    p = m.predict(X_eng) * MAX_VALS[i]
                    p_fold.append(p)
                
                pred_accum += np.column_stack(p_fold)
                folds += 1
                success_folds += 1
            except Exception as e:
                print(f"    Error in fold {fold}: {e}")
        
        if success_folds > 0:
            preds_total += pred_accum / success_folds
            valid_models += 1
            print(f"  -> Success ({success_folds} folds)")

# 4. 平均 & 保存
if valid_models > 0:
    final_pred = preds_total / valid_models
else:
    print("FATAL ERROR: No models worked!")
    final_pred = np.zeros((len(emb_sig), 5))

pred_df = pd.DataFrame(final_pred, columns=TARGET_NAMES)
pred_df["image_path"] = unique_images["image_path"]
pred_df = post_process(pred_df)

sub = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
long_df = pred_df.melt(id_vars=["image_path"], value_vars=TARGET_NAMES, var_name="target_name", value_name="pred")
final = pd.merge(sub[["sample_id", "image_path", "target_name"]], long_df, on=["image_path", "target_name"], how="left")
final["target"] = final["pred"].fillna(0)
final[["sample_id", "target"]].to_csv("submission.csv", index=False)

print("Done! Submission saved.")
print(final.head())